# Day 3: explore the data

Run each cell with **Shift+Enter**. Read the output before moving on, and write down anything surprising.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import poisson



from footiq.data import LEAGUES, load_matches

matches = load_matches('../data/processed/matches.csv')
matches.head()

## 1. How often does each result happen?

H = home win, D = draw, A = away win. Any model must beat just guessing these averages.

In [ ]:
matches['result'].value_counts(normalize=True).round(3)

In [ ]:
by_league = matches.groupby('league')['result'].value_counts(normalize=True).unstack().round(3)
by_league.index = by_league.index.map(LEAGUES)
by_league

## 2. Home advantage over time

Look at 2020/21 (season `2021`): many games were played without fans. Did home advantage shrink?

In [ ]:
home_rate = matches.groupby('season')['result'].apply(lambda r: (r == 'H').mean())
home_rate.plot(kind='bar', title='Home win rate by season', ylim=(0.3, 0.55))
plt.show()

## 3. Goals per match

In [ ]:
matches['total_goals'] = matches['home_goals'] + matches['away_goals']
print('Average home goals:', round(matches['home_goals'].mean(), 3))
print('Average away goals:', round(matches['away_goals'].mean(), 3))
matches.groupby('league')[['home_goals', 'away_goals', 'total_goals']].mean().round(2)

## 4. Does Poisson describe goals well?

The **Poisson distribution** gives the chance of 0, 1, 2, ... events when events happen randomly at some average rate. Goals are rare, random-ish events, so it fits football surprisingly well.

Formula: P(k goals) = rate^k × e^(−rate) / k!

Below, the bars are the real share of matches where the home team scored k goals, and the dots are what Poisson predicts using only the average.

In [ ]:
rate = matches['home_goals'].mean()
k = np.arange(0, 8)
real = matches['home_goals'].value_counts(normalize=True).reindex(k, fill_value=0)

plt.bar(k, real, alpha=0.6, label='real')
plt.plot(k, poisson.pmf(k, rate), 'o-', color='black', label=f'Poisson (rate {rate:.2f})')
plt.xlabel('home goals'); plt.ylabel('share of matches'); plt.legend(); plt.show()
pd.DataFrame({'real': real, 'poisson': poisson.pmf(k, rate)}).round(3)

## 5. The draw problem

If home and away goals were independent Poisson numbers, how many 0-0 and 1-1 draws would we expect? Compare with reality. The gap is exactly what Dixon-Coles fixes on Day 5.

In [ ]:
lam_h, lam_a = matches['home_goals'].mean(), matches['away_goals'].mean()
for h, a in [(0, 0), (1, 1), (1, 0), (0, 1)]:
    real = ((matches['home_goals'] == h) & (matches['away_goals'] == a)).mean()
    expected = poisson.pmf(h, lam_h) * poisson.pmf(a, lam_a)
    print(f'{h}-{a}: real {real:.3f}   independent Poisson {expected:.3f}')

## 6. Your turn

1. Which team scored the most goals at home in the last season?
2. Which league has the most draws?
3. What is the most common scoreline overall? (hint: `value_counts()` on a combined column)

In [ ]:
last = matches[matches['season'] == matches['season'].max()]
last.groupby('home_team')['home_goals'].sum().sort_values(ascending=False).head(10)